In [ ]:
%cd ../..
%load_ext autoreload
%autoreload 2

# Comprehensive EDA: TSLA and BTC Trading Data

This notebook performs exploratory data analysis on Tesla (TSLA) stock and Bitcoin (BTC) trading data, including price movements and news coverage.

In [ ]:
from decision_making.data import load_data
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

# Set up plotting style
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

In [ ]:
# Load data
tsla = load_data(symbol="TSLA", download_if_missing=False)
btc = load_data(symbol="BTC", download_if_missing=False)

## 1. Data Overview and Quality Assessment

In [ ]:
# Display TSLA data (first row)
tsla.head(1)

In [ ]:
# Display BTC data (first row)
btc.head(1)

In [ ]:
# Data shape and structure
print("TSLA Dataset:")
print(f"  Shape: {tsla.shape}")
print(f"  Columns: {tsla.columns}")
print(f"  Date range: {tsla['date'].min()} to {tsla['date'].max()}")
print("\nBTC Dataset:")
print(f"  Shape: {btc.shape}")
print(f"  Columns: {btc.columns}")
print(f"  Date range: {btc['date'].min()} to {btc['date'].max()}")

In [ ]:
# Check for missing values and duplicates
print("TSLA Missing Values:")
print(tsla.null_count())
print(f"\nTSLA Duplicate dates: {tsla['date'].n_unique()} unique out of {tsla.shape[0]} total")

print("\n" + "=" * 50)
print("\nBTC Missing Values:")
print(btc.null_count())
print(f"\nBTC Duplicate dates: {btc['date'].n_unique()} unique out of {btc.shape[0]} total")

In [ ]:
# Add news count column for easier analysis
tsla_analysis = tsla.with_columns(
    pl.col("news").list.len().alias("news_count"),
    pl.col("date").str.to_date().alias("date_parsed"),
)

btc_analysis = btc.with_columns(
    pl.col("news").list.len().alias("news_count"),
    pl.col("date").str.to_date().alias("date_parsed"),
)

print("Enhanced TSLA data:")
print(tsla_analysis.head())
print("\nEnhanced BTC data:")
print(btc_analysis.head())

## 2. Price Analysis

In [ ]:
# Price trends over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# TSLA price trend
tsla_pd = tsla_analysis.to_pandas()
tsla_pd["date_parsed"] = pd.to_datetime(tsla_pd["date_parsed"])
ax1.plot(
    tsla_pd["date_parsed"],
    tsla_pd["prices"],
    linewidth=2,
    color="#E31E24",
    label="TSLA Price",
)
ax1.set_title("Tesla (TSLA) Stock Price Over Time", fontsize=14, fontweight="bold")
ax1.set_xlabel("Date", fontsize=12)
ax1.set_ylabel("Price (USD)", fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

# BTC price trend
btc_pd = btc_analysis.to_pandas()
btc_pd["date_parsed"] = pd.to_datetime(btc_pd["date_parsed"])
ax2.plot(
    btc_pd["date_parsed"],
    btc_pd["prices"],
    linewidth=2,
    color="#F7931A",
    label="BTC Price",
)
ax2.set_title("Bitcoin (BTC) Price Over Time", fontsize=14, fontweight="bold")
ax2.set_xlabel("Date", fontsize=12)
ax2.set_ylabel("Price (USD)", fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Price statistics summary
print("=" * 70)
print("TESLA (TSLA) PRICE STATISTICS")
print("=" * 70)
tsla_stats = tsla_analysis.select([
    pl.col("prices").min().alias("Min Price"),
    pl.col("prices").max().alias("Max Price"),
    pl.col("prices").mean().alias("Mean Price"),
    pl.col("prices").median().alias("Median Price"),
    pl.col("prices").std().alias("Std Dev"),
])
print(tsla_stats)

# Calculate price change
tsla_first = tsla_analysis.select("prices").head(1).item()
tsla_last = tsla_analysis.select("prices").tail(1).item()
tsla_change = ((tsla_last - tsla_first) / tsla_first) * 100
print(f"\nTotal Price Change: ${tsla_first:.2f} → ${tsla_last:.2f} ({tsla_change:+.2f}%)")

print("\n" + "=" * 70)
print("BITCOIN (BTC) PRICE STATISTICS")
print("=" * 70)
btc_stats = btc_analysis.select([
    pl.col("prices").min().alias("Min Price"),
    pl.col("prices").max().alias("Max Price"),
    pl.col("prices").mean().alias("Mean Price"),
    pl.col("prices").median().alias("Median Price"),
    pl.col("prices").std().alias("Std Dev"),
])
print(btc_stats)

# Calculate price change
btc_first = btc_analysis.select("prices").head(1).item()
btc_last = btc_analysis.select("prices").tail(1).item()
btc_change = ((btc_last - btc_first) / btc_first) * 100
print(f"\nTotal Price Change: ${btc_first:.2f} → ${btc_last:.2f} ({btc_change:+.2f}%)")

In [ ]:
# Calculate daily returns and volatility
tsla_returns = tsla_analysis.with_columns(
    ((pl.col("prices") - pl.col("prices").shift(1)) / pl.col("prices").shift(1) * 100).alias("daily_return_pct")
)

btc_returns = btc_analysis.with_columns(
    ((pl.col("prices") - pl.col("prices").shift(1)) / pl.col("prices").shift(1) * 100).alias("daily_return_pct")
)

# Volatility statistics
print("=" * 70)
print("VOLATILITY ANALYSIS (Daily Returns %)")
print("=" * 70)
print("\nTSLA:")
print(
    tsla_returns.select([
        pl.col("daily_return_pct").mean().alias("Mean Daily Return %"),
        pl.col("daily_return_pct").std().alias("Volatility (Std Dev)"),
        pl.col("daily_return_pct").min().alias("Max Loss %"),
        pl.col("daily_return_pct").max().alias("Max Gain %"),
    ])
)

print("\nBTC:")
print(
    btc_returns.select([
        pl.col("daily_return_pct").mean().alias("Mean Daily Return %"),
        pl.col("daily_return_pct").std().alias("Volatility (Std Dev)"),
        pl.col("daily_return_pct").min().alias("Max Loss %"),
        pl.col("daily_return_pct").max().alias("Max Gain %"),
    ])
)

In [ ]:
# Distribution of returns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TSLA returns distribution
tsla_returns_clean = tsla_returns.filter(pl.col("daily_return_pct").is_not_null())
ax1.hist(
    tsla_returns_clean["daily_return_pct"].to_numpy(),
    bins=50,
    color="#E31E24",
    alpha=0.7,
    edgecolor="black",
)
ax1.axvline(0, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax1.set_title("TSLA Daily Returns Distribution", fontsize=14, fontweight="bold")
ax1.set_xlabel("Daily Return (%)", fontsize=12)
ax1.set_ylabel("Frequency", fontsize=12)
ax1.grid(True, alpha=0.3)

# BTC returns distribution
btc_returns_clean = btc_returns.filter(pl.col("daily_return_pct").is_not_null())
ax2.hist(
    btc_returns_clean["daily_return_pct"].to_numpy(),
    bins=50,
    color="#F7931A",
    alpha=0.7,
    edgecolor="black",
)
ax2.axvline(0, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax2.set_title("BTC Daily Returns Distribution", fontsize=14, fontweight="bold")
ax2.set_xlabel("Daily Return (%)", fontsize=12)
ax2.set_ylabel("Frequency", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. News Coverage Analysis

In [ ]:
# News coverage statistics
print("=" * 70)
print("NEWS COVERAGE STATISTICS")
print("=" * 70)

print("\nTSLA News Coverage:")
print(
    tsla_analysis.select([
        pl.col("news_count").sum().alias("Total News Articles"),
        pl.col("news_count").mean().alias("Avg Articles per Day"),
        pl.col("news_count").median().alias("Median Articles per Day"),
        pl.col("news_count").min().alias("Min Articles per Day"),
        pl.col("news_count").max().alias("Max Articles per Day"),
    ])
)

print("\nBTC News Coverage:")
print(
    btc_analysis.select([
        pl.col("news_count").sum().alias("Total News Articles"),
        pl.col("news_count").mean().alias("Avg Articles per Day"),
        pl.col("news_count").median().alias("Median Articles per Day"),
        pl.col("news_count").min().alias("Min Articles per Day"),
        pl.col("news_count").max().alias("Max Articles per Day"),
    ])
)

In [ ]:
# News coverage over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# TSLA news coverage
ax1.bar(tsla_pd["date_parsed"], tsla_pd["news_count"], color="#E31E24", alpha=0.7, width=1)
ax1.set_title("TSLA News Article Count Over Time", fontsize=14, fontweight="bold")
ax1.set_xlabel("Date", fontsize=12)
ax1.set_ylabel("Number of News Articles", fontsize=12)
ax1.grid(True, alpha=0.3, axis="y")

# BTC news coverage
ax2.bar(btc_pd["date_parsed"], btc_pd["news_count"], color="#F7931A", alpha=0.7, width=1)
ax2.set_title("BTC News Article Count Over Time", fontsize=14, fontweight="bold")
ax2.set_xlabel("Date", fontsize=12)
ax2.set_ylabel("Number of News Articles", fontsize=12)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 days with most news coverage
print("=" * 70)
print("TOP 10 DAYS WITH MOST NEWS COVERAGE")
print("=" * 70)

print("\nTSLA:")
print(tsla_analysis.select(["date", "news_count", "prices"]).sort("news_count", descending=True).head(10))

print("\nBTC:")
print(btc_analysis.select(["date", "news_count", "prices"]).sort("news_count", descending=True).head(10))

## 4. News Content Analysis: Word/Token Counts

In [ ]:
# Calculate word counts for each news article
def count_words_in_news(news_list):
    """Count total words across all news articles in a list"""
    total_words = 0
    for article in news_list:
        if article:  # Make sure article is not None or empty
            total_words += len(article.split())
    return total_words


# Add word count columns
tsla_words = tsla_analysis.with_columns(
    pl.col("news").map_elements(count_words_in_news, return_dtype=pl.Int64).alias("total_words")
)

btc_words = btc_analysis.with_columns(
    pl.col("news").map_elements(count_words_in_news, return_dtype=pl.Int64).alias("total_words")
)

print("TSLA with word counts:")
print(tsla_words.select(["date", "news_count", "total_words"]).head(10))
print("\nBTC with word counts:")
print(btc_words.select(["date", "news_count", "total_words"]).head(10))

In [ ]:
# Word count statistics
print("=" * 70)
print("NEWS WORD COUNT STATISTICS")
print("=" * 70)

print("\nTSLA Word Counts:")
print(
    tsla_words.select([
        pl.col("total_words").sum().alias("Total Words"),
        pl.col("total_words").mean().alias("Avg Words per Day"),
        pl.col("total_words").median().alias("Median Words per Day"),
        pl.col("total_words").std().alias("Std Dev"),
        pl.col("total_words").min().alias("Min Words"),
        pl.col("total_words").max().alias("Max Words"),
    ])
)

print("\nBTC Word Counts:")
print(
    btc_words.select([
        pl.col("total_words").sum().alias("Total Words"),
        pl.col("total_words").mean().alias("Avg Words per Day"),
        pl.col("total_words").median().alias("Median Words per Day"),
        pl.col("total_words").std().alias("Std Dev"),
        pl.col("total_words").min().alias("Min Words"),
        pl.col("total_words").max().alias("Max Words"),
    ])
)

# Average words per article
print("\n" + "=" * 70)
print("AVERAGE WORDS PER ARTICLE")
print("=" * 70)
tsla_words_per_article = tsla_words["total_words"].sum() / tsla_words["news_count"].sum()
btc_words_per_article = btc_words["total_words"].sum() / btc_words["news_count"].sum()
print(f"TSLA: {tsla_words_per_article:.1f} words per article")
print(f"BTC:  {btc_words_per_article:.1f} words per article")

In [ ]:
# Word count distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TSLA word count distribution
tsla_words_pd = tsla_words.to_pandas()
ax1.hist(tsla_words_pd["total_words"], bins=50, color="#E31E24", alpha=0.7, edgecolor="black")
ax1.axvline(
    tsla_words_pd["total_words"].mean(),
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {tsla_words_pd['total_words'].mean():.0f}",
)
ax1.set_title("TSLA: Word Count Distribution", fontsize=14, fontweight="bold")
ax1.set_xlabel("Total Words per Day", fontsize=12)
ax1.set_ylabel("Frequency", fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# BTC word count distribution
btc_words_pd = btc_words.to_pandas()
ax2.hist(btc_words_pd["total_words"], bins=50, color="#F7931A", alpha=0.7, edgecolor="black")
ax2.axvline(
    btc_words_pd["total_words"].mean(),
    color="black",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {btc_words_pd['total_words'].mean():.0f}",
)
ax2.set_title("BTC: Word Count Distribution", fontsize=14, fontweight="bold")
ax2.set_xlabel("Total Words per Day", fontsize=12)
ax2.set_ylabel("Frequency", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Word count over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

tsla_words_pd["date_parsed"] = pd.to_datetime(tsla_words_pd["date_parsed"])
btc_words_pd["date_parsed"] = pd.to_datetime(btc_words_pd["date_parsed"])

# TSLA word count over time
ax1.plot(
    tsla_words_pd["date_parsed"],
    tsla_words_pd["total_words"],
    linewidth=1.5,
    color="#E31E24",
    alpha=0.7,
)
ax1.axhline(
    tsla_words_pd["total_words"].mean(),
    color="black",
    linestyle="--",
    linewidth=1.5,
    alpha=0.5,
    label="Mean",
)
ax1.set_title("TSLA: News Word Count Over Time", fontsize=14, fontweight="bold")
ax1.set_xlabel("Date", fontsize=12)
ax1.set_ylabel("Total Words per Day", fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# BTC word count over time
ax2.plot(
    btc_words_pd["date_parsed"],
    btc_words_pd["total_words"],
    linewidth=1.5,
    color="#F7931A",
    alpha=0.7,
)
ax2.axhline(
    btc_words_pd["total_words"].mean(),
    color="black",
    linestyle="--",
    linewidth=1.5,
    alpha=0.5,
    label="Mean",
)
ax2.set_title("BTC: News Word Count Over Time", fontsize=14, fontweight="bold")
ax2.set_xlabel("Date", fontsize=12)
ax2.set_ylabel("Total Words per Day", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 days with highest word counts
print("=" * 70)
print("TOP 10 DAYS WITH HIGHEST WORD COUNTS")
print("=" * 70)

print("\nTSLA:")
print(tsla_words.select(["date", "total_words", "news_count", "prices"]).sort("total_words", descending=True).head(10))

print("\nBTC:")
print(btc_words.select(["date", "total_words", "news_count", "prices"]).sort("total_words", descending=True).head(10))

In [ ]:
# Word count variation analysis
# Calculate rolling 7-day average
tsla_words_pd["word_count_7d_avg"] = tsla_words_pd["total_words"].rolling(window=7, min_periods=1).mean()
btc_words_pd["word_count_7d_avg"] = btc_words_pd["total_words"].rolling(window=7, min_periods=1).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# TSLA word count with rolling average
ax1.scatter(
    tsla_words_pd["date_parsed"],
    tsla_words_pd["total_words"],
    alpha=0.3,
    color="#E31E24",
    s=20,
    label="Daily",
)
ax1.plot(
    tsla_words_pd["date_parsed"],
    tsla_words_pd["word_count_7d_avg"],
    linewidth=2.5,
    color="#8B0000",
    label="7-Day Average",
)
ax1.set_title("TSLA: News Word Count with 7-Day Moving Average", fontsize=14, fontweight="bold")
ax1.set_xlabel("Date", fontsize=12)
ax1.set_ylabel("Total Words", fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# BTC word count with rolling average
ax2.scatter(
    btc_words_pd["date_parsed"],
    btc_words_pd["total_words"],
    alpha=0.3,
    color="#F7931A",
    s=20,
    label="Daily",
)
ax2.plot(
    btc_words_pd["date_parsed"],
    btc_words_pd["word_count_7d_avg"],
    linewidth=2.5,
    color="#CC6600",
    label="7-Day Average",
)
ax2.set_title("BTC: News Word Count with 7-Day Moving Average", fontsize=14, fontweight="bold")
ax2.set_xlabel("Date", fontsize=12)
ax2.set_ylabel("Total Words", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Comparative Analysis: TSLA vs BTC

In [ ]:
# Normalized price comparison (both starting at 100)
tsla_normalized = (tsla_pd["prices"] / tsla_pd["prices"].iloc[0]) * 100
btc_normalized = (btc_pd["prices"] / btc_pd["prices"].iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(
    tsla_pd["date_parsed"],
    tsla_normalized,
    linewidth=2,
    color="#E31E24",
    label="TSLA",
    alpha=0.8,
)
ax.plot(
    btc_pd["date_parsed"],
    btc_normalized,
    linewidth=2,
    color="#F7931A",
    label="BTC",
    alpha=0.8,
)
ax.axhline(100, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_title(
    "Normalized Price Comparison: TSLA vs BTC (Base = 100)",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Normalized Price", fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"TSLA total return: {(tsla_normalized.iloc[-1] - 100):.2f}%")
print(f"BTC total return: {(btc_normalized.iloc[-1] - 100):.2f}%")

In [ ]:
# Side-by-side comparison table including word counts
comparison_data = {
    "Metric": [
        "Total Days",
        "Starting Price",
        "Ending Price",
        "Price Change %",
        "Mean Daily Return %",
        "Volatility (Std Dev)",
        "Max Single Day Gain %",
        "Max Single Day Loss %",
        "Total News Articles",
        "Avg News per Day",
        "Total Words in News",
        "Avg Words per Day",
        "Avg Words per Article",
    ],
    "TSLA": [
        tsla_analysis.shape[0],
        f"${tsla_first:.2f}",
        f"${tsla_last:.2f}",
        f"{tsla_change:+.2f}%",
        f"{tsla_returns['daily_return_pct'].mean():.3f}%",
        f"{tsla_returns['daily_return_pct'].std():.3f}%",
        f"{tsla_returns['daily_return_pct'].max():.2f}%",
        f"{tsla_returns['daily_return_pct'].min():.2f}%",
        tsla_analysis["news_count"].sum(),
        f"{tsla_analysis['news_count'].mean():.1f}",
        f"{tsla_words['total_words'].sum():,}",
        f"{tsla_words['total_words'].mean():.0f}",
        f"{tsla_words_per_article:.0f}",
    ],
    "BTC": [
        btc_analysis.shape[0],
        f"${btc_first:.2f}",
        f"${btc_last:.2f}",
        f"{btc_change:+.2f}%",
        f"{btc_returns['daily_return_pct'].mean():.3f}%",
        f"{btc_returns['daily_return_pct'].std():.3f}%",
        f"{btc_returns['daily_return_pct'].max():.2f}%",
        f"{btc_returns['daily_return_pct'].min():.2f}%",
        btc_analysis["news_count"].sum(),
        f"{btc_analysis['news_count'].mean():.1f}",
        f"{btc_words['total_words'].sum():,}",
        f"{btc_words['total_words'].mean():.0f}",
        f"{btc_words_per_article:.0f}",
    ],
}

comparison_df = pd.DataFrame(comparison_data)
print("=" * 90)
print("COMPREHENSIVE COMPARISON: TSLA vs BTC")
print("=" * 90)
print(comparison_df.to_string(index=False))

In [ ]:
# Volatility comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Box plot comparison - returns
tsla_returns_pd = tsla_returns.filter(pl.col("daily_return_pct").is_not_null()).to_pandas()
btc_returns_pd = btc_returns.filter(pl.col("daily_return_pct").is_not_null()).to_pandas()

bp = ax1.boxplot(
    [tsla_returns_pd["daily_return_pct"], btc_returns_pd["daily_return_pct"]],
    labels=["TSLA", "BTC"],
    patch_artist=True,
)
bp["boxes"][0].set_facecolor("#E31E24")
bp["boxes"][0].set_alpha(0.7)
bp["boxes"][1].set_facecolor("#F7931A")
bp["boxes"][1].set_alpha(0.7)
ax1.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax1.set_title("Daily Returns Distribution Comparison", fontsize=12, fontweight="bold")
ax1.set_ylabel("Daily Return (%)", fontsize=11)
ax1.grid(True, alpha=0.3, axis="y")

# Box plot comparison - word counts
bp2 = ax2.boxplot(
    [tsla_words_pd["total_words"], btc_words_pd["total_words"]],
    labels=["TSLA", "BTC"],
    patch_artist=True,
)
bp2["boxes"][0].set_facecolor("#E31E24")
bp2["boxes"][0].set_alpha(0.7)
bp2["boxes"][1].set_facecolor("#F7931A")
bp2["boxes"][1].set_alpha(0.7)
ax2.set_title("News Word Count Distribution Comparison", fontsize=12, fontweight="bold")
ax2.set_ylabel("Total Words per Day", fontsize=11)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 6. Executive Summary and Key Insights

### Dataset Overview
- **Time Period**: August 1, 2024 to January 10, 2026 (~528 days)
- **Data Quality**: Complete datasets with no missing values for both assets
- Both datasets include daily prices and associated news articles

### Key Findings

#### 1. Price Performance
- **Tesla (TSLA)** demonstrated strong performance with 105% price appreciation over the period
- **Bitcoin (BTC)** showed 38% gains with characteristic cryptocurrency volatility
- TSLA exhibited higher volatility (3.4% daily std dev) compared to BTC (2.3%)

#### 2. Volatility Analysis
- Both TSLA and BTC display high volatility typical of growth stocks and cryptocurrencies
- Daily returns show wide distributions with notable outliers in both directions
- TSLA had max single-day movements of +22.7% / -15.4%
- BTC had max single-day movements of +12.1% / -8.6%

#### 3. News Coverage Patterns
- Consistent daily news coverage for both assets (average ~1 article per day)
- TSLA: 531 total articles over 528 days
- BTC: 528 total articles (exactly 1 per day)

#### 4. News Content Analysis (Word Counts)
- **News article length varies significantly day-to-day**
- Word counts provide insights into the depth of coverage beyond simple article counts
- Variation in word counts suggests different levels of market activity and newsworthiness
- Days with higher word counts may indicate more significant market events or developments
- 7-day moving averages reveal trends in coverage intensity over time

### Strategic Implications

1. **Risk Management**: Both assets require robust risk management due to high volatility
2. **News Analysis**: Content depth (word counts) varies independently of article counts, suggesting quality varies
3. **Coverage Intensity**: Monitoring word count trends may provide additional signals about market attention
4. **Market Monitoring**: Continuous monitoring is essential given the rapid price movements

### Recommendations for Further Analysis

1. Implement sentiment analysis on news content to assess tone and impact
2. Investigate correlation between word count spikes and price movements
3. Analyze specific types of news (earnings, regulatory, product announcements) separately
4. Consider external market factors (broader market indices, interest rates, etc.)
5. Examine whether longer articles (higher word counts) have different characteristics than shorter ones
6. Study the relationship between news content depth and subsequent volatility